# Plant Disease Identification Model Training

## Import Libraries

In [1]:
import os
import gc
import pickle
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten,
                                     Dense, Dropout, BatchNormalization)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print(" All libraries imported successfully.")

✅ All libraries imported successfully.


## Model Training

In [2]:
# ── UPDATE THIS PATH ─────────────────────────────────────────────────────────
dataset_path = r"D:\VS Code\Python\plant disease idetification\plantvillage dataset\color"
# ─────────────────────────────────────────────────────────────────────────────

IMAGE_SIZE  = 32
BATCH_SIZE  = 32
SAMPLE_SIZE = 5000

# ── Rebuild generators & PCA (needed if running this file standalone) ─────────
datagen_train = ImageDataGenerator(rescale=1./255, validation_split=0.2,
    rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
    horizontal_flip=True, zoom_range=0.1, fill_mode='nearest')
datagen_val = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = datagen_train.flow_from_directory(dataset_path,
    target_size=(IMAGE_SIZE,IMAGE_SIZE), batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', shuffle=True, seed=42)
val_gen = datagen_val.flow_from_directory(dataset_path,
    target_size=(IMAGE_SIZE,IMAGE_SIZE), batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', shuffle=False, seed=42)

num_classes = train_gen.num_classes
class_names = list(train_gen.class_indices.keys())

encoder = LabelEncoder()
encoder.fit(class_names)

datagen_flat = ImageDataGenerator(rescale=1./255)
flat_gen = datagen_flat.flow_from_directory(dataset_path,
    target_size=(IMAGE_SIZE,IMAGE_SIZE), batch_size=BATCH_SIZE,
    class_mode='sparse', shuffle=True, seed=42)
sample_x, sample_y = [], []
for imgs, lbls in flat_gen:
    sample_x.append(imgs); sample_y.append(lbls)
    if sum(len(b) for b in sample_x) >= SAMPLE_SIZE: break
sample_x = np.concatenate(sample_x)[:SAMPLE_SIZE]
sample_y = np.concatenate(sample_y)[:SAMPLE_SIZE].astype(int)
X_flat = sample_x.reshape(len(sample_x), -1).astype(np.float32)
del sample_x; gc.collect()
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_flat, sample_y, test_size=0.2, random_state=42, stratify=sample_y)
del X_flat; gc.collect()
pca = PCA(n_components=100, random_state=42)
X_train_pca = pca.fit_transform(X_train_s)
X_test_pca  = pca.transform(X_test_s)
del X_train_s, X_test_s; gc.collect()
print("Setup complete. Starting model training...")

Found 49961 images belonging to 38 classes.
Found 12476 images belonging to 38 classes.
Found 62437 images belonging to 38 classes.
✅ Setup complete. Starting model training...


### SVM

In [3]:
print("Training SVM on 5,000-sample PCA features ...")
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_pca, y_train_s)
print("✅ SVM training complete.")

Training SVM on 5,000-sample PCA features ...
✅ SVM training complete.


### Random Forest

In [4]:
print("Training Random Forest ...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_pca, y_train_s)
print(" Random Forest training complete.")

Training Random Forest ...
✅ Random Forest training complete.


### CNN 

In [5]:
gc.collect()
tf.keras.backend.clear_session()

model = Sequential([
    # Block 1
    Conv2D(32, (3,3), activation='relu', padding='same',
           input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    # Block 2
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    # Block 3
    Conv2D(128, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)),

    # Head
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.4),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer = 'adam',
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)
model.summary()

C:\Users\Vaseem\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 32, 32, 32)          │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 32, 32, 32)          │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 16, 16, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 16, 16, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 16, 16, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 8, 8, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 8, 8, 128)           │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 4, 4, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 2048)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         524,544 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 38)                  │           9,766 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 627,942 (2.40 MB)

 Trainable params: 627,750 (2.39 MB)

 Non-trainable params: 192 (768.00 B)

In [6]:
history = model.fit(
    train_gen,
    epochs          = 10,
    validation_data = val_gen,
    verbose         = 1
)
print("\n CNN training complete.")

Epoch 1/10
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 138s 87ms/step - accuracy: 0.6104 - loss: 1.3395 - val_accuracy: 0.5345 - val_loss: 1.9811
Epoch 2/10
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 126s 81ms/step - accuracy: 0.7664 - loss: 0.7474 - val_accuracy: 0.7540 - val_loss: 0.8168
Epoch 3/10
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 134s 86ms/step - accuracy: 0.8145 - loss: 0.5926 - val_accuracy: 0.6992 - val_loss: 1.0610
Epoch 4/10
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 122s 78ms/step - accuracy: 0.8414 - loss: 0.5007 - val_accuracy: 0.8140 - val_loss: 0.6090
Epoch 5/10
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 116s 74ms/step - accuracy: 0.8590 - loss: 0.4419 - val_accuracy: 0.6573 - val_loss: 1.8204
Epoch 6/10
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 123s 79ms/step - accuracy: 0.8724 - loss: 0.4069 - val_accuracy: 0.8627 - val_loss: 0.4739
Epoch 7/10
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 124s 79ms/step - accuracy: 0.8829 - loss: 0.3755 - val_accuracy: 0.6742 - val_loss: 1.5316
Epoch 8/10
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 134s 86ms/step - accuracy: 

In [8]:
svm_pred     = svm_model.predict(X_test_pca)
svm_accuracy = accuracy_score(y_test_s, svm_pred)
rf_pred      = rf_model.predict(X_test_pca)
rf_accuracy  = accuracy_score(y_test_s, rf_pred)
cnn_loss, cnn_accuracy = model.evaluate(val_gen, verbose=0)

y_pred_cnn, y_true_cnn = [], []
val_gen.reset()
for imgs, lbls in val_gen:
    preds = model.predict(imgs, verbose=0)
    y_pred_cnn.extend(np.argmax(preds, axis=1))
    y_true_cnn.extend(np.argmax(lbls,  axis=1))
    if len(y_true_cnn) >= val_gen.n:
        break
y_pred_cnn = np.array(y_pred_cnn[:val_gen.n])
y_true_cnn = np.array(y_true_cnn[:val_gen.n])

print(f"SVM Accuracy           : {svm_accuracy*100:.2f}%  (on 5000-sample subset)")
print(f"Random Forest Accuracy : {rf_accuracy*100:.2f}%  (on 5000-sample subset)")
print(f"CNN Accuracy           : {cnn_accuracy*100:.2f}%  (on full validation set)")
print("\n Model Evaluation complete.")

SVM Accuracy           : 64.50%  (on 5000-sample subset)
Random Forest Accuracy : 51.80%  (on 5000-sample subset)
CNN Accuracy           : 86.38%  (on full validation set)

✅ Model Evaluation complete.
